# B.O.S.S. — Test EfficientDet Lite4 (TFLite) su recordings

**Pipeline:**
1. Ricerca frame (dataset gia estratto su Kaggle)
2. Caricamento EfficientDet Lite4 `.tflite` + inferenza -> frame annotati
3. Distribuzione classi e confidenza
4. Caricamento Ground Truth (export Roboflow **COCO JSON**)
5. Valutazione quantitativa contro GT (mAP, Precision*, Recall* via torchmetrics)
6. Metriche per classe + grafici
7. Griglia campione frame + dashboard riepilogo
8. Report Markdown leggibile da IA

---
**Output - `/kaggle/working/TEST_OUTPUT/` (ricreata da zero a ogni run):**
- `model_output/` - predizioni sui recordings: frame annotati, `predictions.csv`, distribuzione classi/confidenza, frame campione
- `gt_eval/` - grafici valutazione contro Ground Truth (AP per classe, metriche CSV)
- `comparison/` - dashboard di confronto GT vs output modello
- `markdown/` - report `.md` riepilogativi

---
**Kaggle - dati di input:**
- **Dataset** recordings: cartella `images`
- **Dataset** GT: export Roboflow **COCO** con `train/test/valid` (immagini + `_annotations.coco.json` in ogni split)
- **Model**: EfficientDet Lite4 `2.tflite`

In [ ]:
# ============================================================
# Cella 1 - Import librerie
# ============================================================
%pip install torchmetrics --quiet

import os
import shutil
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
from torchmetrics.detection import MeanAveragePrecision
import tensorflow as tf

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Cella 2 - Configurazione + mappa classi universale (BOSS <-> COCO)
# ============================================================
import re

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

if IS_KAGGLE:
    # -- Kaggle -----------------------------------------------
    EFFICIENTDET_PATH = Path("/kaggle/input/models/tensorflow/efficientdet/tflite/lite4-detection-default/2/2.tflite")
    RECORDINGS_DIR    = Path("/kaggle/input/datasets/lorenzoverdura/boss-recordings/images")
    GT_COCO_DIR       = Path("/kaggle/input/boss-gt-coco")
    BASE_DIR          = Path("/kaggle/input")
else:
    # -- Locale -----------------------------------------------
    BASE_DIR          = Path("/home/lorenzoverdura8/BOSS_CODE")
    EFFICIENTDET_PATH = BASE_DIR / "2.tflite"
    RECORDINGS_DIR    = BASE_DIR / "recordings"
    GT_COCO_DIR       = BASE_DIR / "gt_coco"

OUTPUT_DIR     = Path("/kaggle/working")
TEST_OUTPUT    = OUTPUT_DIR / "TEST_OUTPUT"
DIR_MODEL_OUT  = TEST_OUTPUT / "model_output"
DIR_GT_EVAL    = TEST_OUTPUT / "gt_eval"
DIR_COMPARISON = TEST_OUTPUT / "comparison"
DIR_MARKDOWN   = TEST_OUTPUT / "markdown"

BOSS_CLASSES = [
    "Bench", "Bicycle Rack", "Bike", "Car", "Chair", "Dustbin",
    "Electrical Box", "Electrical Pole", "Manhole", "Motorcycle",
    "Pedestrian crosswalk", "Person", "Plant Pot", "Road", "Stairs",
    "Table", "Teraffic Barrel", "Traffic sign", "Tree", "Truck",
]
NUM_CLASSES = len(BOSS_CLASSES)

BOSS_ALIASES = {
    "Bench":                ["bench"],
    "Bicycle Rack":         ["bicycle rack", "bike rack", "cycle rack"],
    "Bike":                 ["bike", "bicycle", "cycle"],
    "Car":                  ["car", "automobile"],
    "Chair":                ["chair"],
    "Dustbin":              ["dustbin", "bin", "trash can", "trashcan", "garbage can", "waste bin", "trash"],
    "Electrical Box":       ["electrical box", "electric box", "junction box", "utility box"],
    "Electrical Pole":      ["electrical pole", "electric pole", "utility pole", "power pole", "pole"],
    "Manhole":              ["manhole", "manhole cover"],
    "Motorcycle":           ["motorcycle", "motorbike", "motor bike"],
    "Pedestrian crosswalk": ["pedestrian crosswalk", "crosswalk", "cross walk", "zebra crossing", "pedestrian crossing"],
    "Person":               ["person", "pedestrian", "people", "human"],
    "Plant Pot":            ["plant pot", "potted plant", "pot plant", "flower pot", "flowerpot", "planter"],
    "Road":                 ["road", "street", "roadway"],
    "Stairs":               ["stairs", "staircase", "steps", "stair"],
    "Table":                ["table", "dining table", "desk"],
    "Teraffic Barrel":      ["teraffic barrel", "traffic barrel", "barrel", "traffic drum", "construction barrel"],
    "Traffic sign":         ["traffic sign", "road sign", "street sign", "stop sign", "traffic signal"],
    "Tree":                 ["tree"],
    "Truck":                ["truck", "lorry"],
}

def normalize_name(name):
    return re.sub(r"[\s_\-]+", " ", str(name).strip().lower())

ALIAS_TO_BOSS = {}
for _boss in BOSS_CLASSES:
    ALIAS_TO_BOSS[normalize_name(_boss)] = _boss
    for _alias in BOSS_ALIASES.get(_boss, []):
        ALIAS_TO_BOSS[normalize_name(_alias)] = _boss

def resolve_to_boss(name):
    return ALIAS_TO_BOSS.get(normalize_name(name))

def build_model_to_boss(model_names):
    mapping = {}
    for mid, mname in model_names.items():
        boss = resolve_to_boss(mname)
        if boss is not None:
            mapping[int(mid)] = BOSS_CLASSES.index(boss)
    return mapping

# Classi COCO originali 1-based (con gap: mancano 12, 26, 29, 30, ...).
# COCO_ID_TO_SEQ converte coco_id 1-based -> indice 0-based sequenziale
# affinche MODEL_CLASSES sia un dict {0..79: nome} compatibile con le celle downstream.
_COCO_RAW = {
    1:"person", 2:"bicycle", 3:"car", 4:"motorcycle", 5:"airplane",
    6:"bus", 7:"train", 8:"truck", 9:"boat", 10:"traffic light",
    11:"fire hydrant", 13:"stop sign", 14:"parking meter", 15:"bench",
    16:"bird", 17:"cat", 18:"dog", 19:"horse", 20:"sheep",
    21:"cow", 22:"elephant", 23:"bear", 24:"zebra", 25:"giraffe",
    27:"backpack", 28:"umbrella", 31:"handbag", 32:"tie", 33:"suitcase",
    34:"frisbee", 35:"skis", 36:"snowboard", 37:"sports ball", 38:"kite",
    39:"baseball bat", 40:"baseball glove", 41:"skateboard", 42:"surfboard",
    43:"tennis racket", 44:"bottle", 46:"wine glass", 47:"cup",
    48:"fork", 49:"knife", 50:"spoon", 51:"bowl", 52:"banana",
    53:"apple", 54:"sandwich", 55:"orange", 56:"broccoli", 57:"carrot",
    58:"hot dog", 59:"pizza", 60:"donut", 61:"cake", 62:"chair",
    63:"couch", 64:"potted plant", 65:"bed", 67:"dining table",
    70:"toilet", 72:"tv", 73:"laptop", 74:"mouse", 75:"remote",
    76:"keyboard", 77:"cell phone", 78:"microwave", 79:"oven",
    80:"toaster", 81:"sink", 82:"refrigerator", 84:"book",
    85:"clock", 86:"vase", 87:"scissors", 88:"teddy bear",
    89:"hair drier", 90:"toothbrush",
}

_coco_keys_sorted = sorted(_COCO_RAW.keys())
COCO_ID_TO_SEQ    = {cid: idx for idx, cid in enumerate(_coco_keys_sorted)}  # 1-based -> 0-based
SEQ_TO_COCO_ID    = {v: k for k, v in COCO_ID_TO_SEQ.items()}                # 0-based -> 1-based

MODEL_CLASSES = {COCO_ID_TO_SEQ[k]: v for k, v in _COCO_RAW.items()}  # {0:"person",...,79:"toothbrush"}
MODEL_NC      = len(MODEL_CLASSES)   # 80
MODEL_TO_BOSS = build_model_to_boss(MODEL_CLASSES)   # {seq_id: boss_id}

IMG_SIZE       = 640
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.45
DEVICE         = "cpu"               # TFLite usa CPU
MODEL_PATH     = EFFICIENTDET_PATH   # alias per celle dashboard/report

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if TEST_OUTPUT.exists():
    shutil.rmtree(TEST_OUTPUT)
for d in (DIR_MODEL_OUT, DIR_GT_EVAL, DIR_COMPARISON, DIR_MARKDOWN):
    d.mkdir(parents=True, exist_ok=True)

mapped_names = sorted({BOSS_CLASSES[b] for b in MODEL_TO_BOSS.values()})
missing      = [c for c in BOSS_CLASSES if c not in mapped_names]
print(f"Ambiente:           {'Kaggle' if IS_KAGGLE else 'Locale'}")
print(f"EFFICIENTDET_PATH:  {EFFICIENTDET_PATH}  -  esiste: {EFFICIENTDET_PATH.exists()}")
print(f"RECORDINGS_DIR:     {RECORDINGS_DIR}  -  esiste: {RECORDINGS_DIR.exists()}")
print(f"GT_COCO_DIR:        {GT_COCO_DIR}  -  esiste: {GT_COCO_DIR.exists()}")
print(f"Classi COCO (seq):  {MODEL_NC}")
print(f"Classi BOSS coperte ({len(mapped_names)}/{NUM_CLASSES}): {mapped_names}")
print(f"Classi BOSS NON coperte: {missing}")
print(f"CONF_THRESHOLD:     {CONF_THRESHOLD}")
print(f"IOU_THRESHOLD:      {IOU_THRESHOLD}")

In [ ]:
# ============================================================
# Cella 3 - Ricerca frame recordings (Kaggle: dataset gia estratto)
# ============================================================
# Su Kaggle il dataset e gia estratto in sola lettura sotto /kaggle/input.
# Le immagini vengono consolidate in un'unica cartella scrivibile in
# /kaggle/working (via symlink, senza copia), usata come sorgente per
# l'inferenza (Cella 5).

recordings_root = RECORDINGS_DIR
if not recordings_root.exists():
    raise FileNotFoundError(f"Cartella recordings non trovata: {recordings_root}")

TEST_FRAMES_DIR = OUTPUT_DIR / "frames_all"
if TEST_FRAMES_DIR.exists():
    shutil.rmtree(TEST_FRAMES_DIR)
TEST_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

img_files = sorted(
    p for p in recordings_root.rglob("*")
    if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
if not img_files:
    raise FileNotFoundError(
        f"Nessuna immagine (.jpg/.jpeg/.png) trovata sotto {recordings_root}"
    )

all_frames = []
for img in img_files:
    dst = TEST_FRAMES_DIR / img.name
    try:
        dst.symlink_to(img.resolve())
    except OSError:
        shutil.copy(img, dst)
    all_frames.append(dst)

print(f"Totale frame trovati: {len(all_frames)}")
print(f"TEST_FRAMES_DIR (consolidata): {TEST_FRAMES_DIR}")

In [ ]:
# ============================================================
# Cella 4 - EfficientDet TFLite: interpreter + run_efficientdet()
# ============================================================
interpreter = tf.lite.Interpreter(model_path=str(EFFICIENTDET_PATH))
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

INPUT_SHAPE = input_details[0]["shape"]
INPUT_H, INPUT_W = int(INPUT_SHAPE[1]), int(INPUT_SHAPE[2])
INPUT_DTYPE = input_details[0]["dtype"]

print(f"Input:  {INPUT_H}x{INPUT_W}  dtype={INPUT_DTYPE.__name__}")
print("Output tensors:")
for d in output_details:
    print(f"  idx={d['index']}  shape={tuple(d['shape'])}  dtype={d['dtype'].__name__}  name='{d['name']}'")

# Identificazione per shape - robusto indipendentemente dai nomi dei tensori.
# EfficientDet Lite TFLite (default): class ID 0-based (0=person,1=bicycle,2=car,...)
#   [1, N, 4]  -> detection_boxes
#   [1, N]     -> detection_classes  (primo tra i 2D)
#   [1, N]     -> detection_scores   (secondo tra i 2D)
#   [1]        -> num_detections
IDX_BOXES = IDX_CLASSES = IDX_SCORES = IDX_NUM = None
_two_dim = []

for d in sorted(output_details, key=lambda x: x["index"]):
    s = tuple(d["shape"])
    if len(s) == 3 and s[-1] == 4:
        IDX_BOXES = d["index"]
    elif s == (1,) or s == (1, 1):
        IDX_NUM = d["index"]
    elif len(s) == 2 and s[0] == 1 and s[1] > 1:
        _two_dim.append(d["index"])

if len(_two_dim) >= 2:
    IDX_CLASSES, IDX_SCORES = _two_dim[0], _two_dim[1]
elif len(_two_dim) == 1:
    IDX_CLASSES = IDX_SCORES = _two_dim[0]

print(f"\nMappatura: boxes={IDX_BOXES}  classes={IDX_CLASSES}  scores={IDX_SCORES}  num={IDX_NUM}")
assert None not in (IDX_BOXES, IDX_CLASSES, IDX_SCORES, IDX_NUM), \
    "Impossibile identificare tutti i tensori di output - controlla i log sopra"


def run_efficientdet(img_bgr):
    """
    Inferenza su immagine BGR numpy. Restituisce (boxes_xyxy_abs, seq_ids, scores).
    EfficientDet Lite TFLite usa class ID 0-based -> +1 per convertire a COCO 1-based
    prima della lookup in COCO_ID_TO_SEQ.
    """
    h_orig, w_orig = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inp = cv2.resize(img_rgb, (INPUT_W, INPUT_H)).astype(INPUT_DTYPE)
    if INPUT_DTYPE == np.float32:
        inp = inp / 255.0
    interpreter.set_tensor(input_details[0]["index"], inp[np.newaxis])
    interpreter.invoke()

    n_det   = int(interpreter.get_tensor(IDX_NUM).flat[0])
    boxes   = interpreter.get_tensor(IDX_BOXES)[0][:n_det]    # [N,4] ymin xmin ymax xmax norm
    classes = interpreter.get_tensor(IDX_CLASSES)[0][:n_det]  # [N] float, 0-based
    scores  = interpreter.get_tensor(IDX_SCORES)[0][:n_det]   # [N]

    ymin, xmin, ymax, xmax = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    boxes_xyxy = np.stack([xmin*w_orig, ymin*h_orig, xmax*w_orig, ymax*h_orig], axis=1)

    # EfficientDet Lite: class ID 0-based -> +1 -> COCO 1-based -> lookup seq_id
    coco_ids = classes.astype(int) + 1
    seq_ids  = np.array([COCO_ID_TO_SEQ.get(int(c), -1) for c in coco_ids])

    return boxes_xyxy, seq_ids, scores


# -- Debug: prima inferenza per verificare i valori grezzi -----------------
_dbg_img = next(
    (p for p in TEST_FRAMES_DIR.iterdir() if p.suffix.lower() in (".jpg",".jpeg",".png")),
    None
)
if _dbg_img:
    _b, _s, _sc = run_efficientdet(cv2.imread(str(_dbg_img)))
    print(f"\nDebug prima immagine ({_dbg_img.name}):")
    print(f"  n valide (sid>=0): {(_s>=0).sum()}")
    print(f"  class seq_ids raw: {_s[:10]}")
    print(f"  scores raw:        {np.round(_sc[:10],3)}")

In [ ]:
# ============================================================
# Cella 5 - Inferenza su tutti i frame -> df_preds + frame annotati
# ============================================================
all_predictions  = []
PREDICT_SAVE_DIR = DIR_MODEL_OUT / "annotated_frames"
PREDICT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

frame_list = sorted(
    p for p in TEST_FRAMES_DIR.iterdir()
    if p.suffix.lower() in (".jpg", ".jpeg", ".png")
)

for img_path in tqdm(frame_list, desc="EfficientDet inference"):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    frame_name = img_path.name
    boxes_xyxy, seq_ids, scores = run_efficientdet(img_bgr)
    ann = img_bgr.copy()

    for box, sid, score in zip(boxes_xyxy, seq_ids, scores):
        if score < CONF_THRESHOLD or sid < 0:
            continue
        boss_id    = MODEL_TO_BOSS.get(int(sid))
        class_name = MODEL_CLASSES.get(int(sid), str(sid))
        all_predictions.append({
            "frame":      frame_name,
            "class_id":   int(sid),
            "class_name": class_name,
            "boss_class": BOSS_CLASSES[boss_id] if boss_id is not None else None,
            "confidence": float(score),
            "x1": float(box[0]), "y1": float(box[1]),
            "x2": float(box[2]), "y2": float(box[3]),
        })
        cv2.rectangle(ann, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0,255,0), 2)
        cv2.putText(ann, f"{class_name} {score:.2f}",
                    (int(box[0]), max(int(box[1])-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    cv2.imwrite(str(PREDICT_SAVE_DIR / frame_name), ann)

df_preds = pd.DataFrame(all_predictions)
print(f"\nFrame processati:  {len(frame_list)}")
print(f"Predizioni totali: {len(df_preds)}")
print(f"Frame con pred.:   {df_preds['frame'].nunique() if not df_preds.empty else 0}")
print(f"Annotati in:       {PREDICT_SAVE_DIR}")
df_preds.to_csv(DIR_MODEL_OUT / "predictions.csv", index=False)
if not df_preds.empty:
    print(df_preds.head(10).to_string(index=False))

In [ ]:
# ============================================================
# Cella 6 - Distribuzione classi (BOSS) e confidenza
# ============================================================
if df_preds.empty:
    print("Nessuna predizione - grafici saltati.")
else:
    # Grafico 1: rilevamenti per classe BOSS - ordine fisso = BOSS_CLASSES
    df_boss = df_preds.dropna(subset=["boss_class"])
    class_counts = (
        df_boss["boss_class"]
        .value_counts()
        .reindex(BOSS_CLASSES, fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = plt.cm.tab20.colors[:NUM_CLASSES]
    ax.bar(class_counts.index, class_counts.values, color=colors)
    ax.set_title("Distribuzione classi BOSS rilevate - recordings", fontsize=14)
    ax.set_xlabel("Classe BOSS")
    ax.set_ylabel("Numero rilevamenti")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_class_distribution.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

    # Grafico 2: distribuzione confidenza (tutte le predizioni)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_preds["confidence"], bins=50, color="steelblue", edgecolor="white")
    ax.axvline(
        CONF_THRESHOLD, color="red", linestyle="--",
        label=f"soglia={CONF_THRESHOLD}"
    )
    ax.set_title("Distribuzione confidenza predizioni", fontsize=13)
    ax.set_xlabel("Confidenza")
    ax.set_ylabel("Frequenza")
    ax.legend()
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_confidence_distribution.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 7 - Ground Truth (COCO JSON Roboflow) + valutazione torchmetrics
# ============================================================
# Carica le annotazioni GT in formato COCO (export Roboflow) e calcola
# mAP@0.5, mAP@0.5:0.95 con torchmetrics. Precision e Recall per classe
# sono calcolate in modo reale via matching IoU (TP/FP/FN) a IoU 0.5 e
# soglia di confidenza CONF_THRESHOLD. Le bbox COCO sono [x, y, w, h]
# assolute in pixel.

# COCO Roboflow: immagini direttamente in test/, NON in test/images/
gt_json_path = GT_COCO_DIR / "test" / "_annotations.coco.json"
GT_TEST_IMGS = GT_COCO_DIR / "test"

if not gt_json_path.exists():
    raise FileNotFoundError(f"Annotazioni COCO non trovate: {gt_json_path}")

with open(gt_json_path) as f:
    gt_data = json.load(f)

print(f"GT COCO: {len(gt_data['images'])} immagini, "
      f"{len(gt_data['annotations'])} annotazioni, "
      f"{len(gt_data['categories'])} categorie")

# category_id (Roboflow) -> seq_id EfficientDet (0-based), via classe BOSS.
# Le categorie GT non mappabili su una classe COCO rilevabile vengono scartate.
cat_id_to_seq = {}
unmapped_cats = []
for cat in gt_data["categories"]:
    boss = resolve_to_boss(cat["name"])
    seq = None
    if boss is not None:
        for seq_id, coco_name in MODEL_CLASSES.items():
            if resolve_to_boss(coco_name) == boss:
                seq = seq_id
                break
    if seq is None:
        unmapped_cats.append(cat["name"])
    else:
        cat_id_to_seq[cat["id"]] = seq
print(f"Categorie GT mappate (cat_id -> seq_id): {cat_id_to_seq}")
print(f"Categorie GT scartate (non rilevabili dal modello): {unmapped_cats}")

# image_id -> lista annotazioni
img_id_to_anns = {}
for ann in gt_data["annotations"]:
    img_id_to_anns.setdefault(ann["image_id"], []).append(ann)

metric_50   = MeanAveragePrecision(iou_type="bbox", class_metrics=True, iou_thresholds=[0.5])
metric_full = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

# Accumulatori TP/FP/FN per classe (seq_id) per Precision/Recall reali.
MATCH_IOU = 0.5
tp_pc = np.zeros(MODEL_NC, dtype=np.int64)
fp_pc = np.zeros(MODEL_NC, dtype=np.int64)
fn_pc = np.zeros(MODEL_NC, dtype=np.int64)

def _iou_one_to_many(box, boxes):
    """IoU tra una box [x1,y1,x2,y2] e un array [M,4]."""
    if len(boxes) == 0:
        return np.zeros((0,), dtype=np.float64)
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    iw = np.clip(x2 - x1, 0, None)
    ih = np.clip(y2 - y1, 0, None)
    inter = iw * ih
    area_b = (box[2] - box[0]) * (box[3] - box[1])
    area_o = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (area_b + area_o - inter + 1e-9)

processed = 0
for img_info in tqdm(gt_data["images"], desc="Valutazione GT (COCO)"):
    img_path = GT_TEST_IMGS / img_info["file_name"]
    img_bgr  = cv2.imread(str(img_path))
    if img_bgr is None:
        continue

    gt_boxes, gt_labels = [], []
    for ann in img_id_to_anns.get(img_info["id"], []):
        sid = cat_id_to_seq.get(ann["category_id"])
        if sid is None:
            continue
        x, y, bw, bh = ann["bbox"]      # COCO: assoluto, xywh
        gt_boxes.append([x, y, x + bw, y + bh])
        gt_labels.append(sid)
    if not gt_boxes:
        continue

    targets = [{"boxes":  torch.tensor(gt_boxes,  dtype=torch.float32),
                "labels": torch.tensor(gt_labels, dtype=torch.int64)}]

    boxes_xyxy, seq_ids, scores = run_efficientdet(img_bgr)
    valid = seq_ids >= 0
    preds = [{"boxes":  torch.tensor(boxes_xyxy[valid], dtype=torch.float32),
              "scores": torch.tensor(scores[valid],     dtype=torch.float32),
              "labels": torch.tensor(seq_ids[valid],    dtype=torch.int64)}]

    metric_50.update(preds, targets)
    metric_full.update(preds, targets)

    # --- TP/FP/FN per classe: greedy matching @ conf CONF_THRESHOLD, IoU MATCH_IOU ---
    keep       = valid & (scores >= CONF_THRESHOLD)
    p_boxes    = boxes_xyxy[keep]
    p_labels   = seq_ids[keep].astype(np.int64)
    p_scores   = scores[keep]
    gt_arr     = np.array(gt_boxes,  dtype=np.float64)
    gt_lab_arr = np.array(gt_labels, dtype=np.int64)

    for cls in np.unique(np.concatenate([p_labels, gt_lab_arr])):
        cls = int(cls)
        pb = p_boxes[p_labels == cls]
        ps = p_scores[p_labels == cls]
        gb = gt_arr[gt_lab_arr == cls]
        matched = np.zeros(len(gb), dtype=bool)
        for pi in np.argsort(-ps):          # predizioni per confidenza decrescente
            ious = _iou_one_to_many(pb[pi], gb)
            if len(ious):
                ious = ious.copy()
                ious[matched] = -1.0
                best = int(ious.argmax())
                if ious[best] >= MATCH_IOU:
                    matched[best] = True
                    tp_pc[cls] += 1
                    continue
            fp_pc[cls] += 1
        fn_pc[cls] += int((~matched).sum())

    processed += 1

print(f"Immagini valutate: {processed}")

res_50   = metric_50.compute()
res_full = metric_full.compute()

map50   = float(res_50["map"])
map5095 = float(res_full["map"])

# Filtra classi con AP=-1 (nessun match GT o pred)
classes_t = res_50["classes"]
ap50_t    = res_50["map_per_class"]
ap5095_t  = res_full["map_per_class"]

valid_cls           = (ap50_t >= 0) & (ap5095_t >= 0)
gt_class_indices    = classes_t[valid_cls].numpy().astype(int)
gt_per_class_ap50   = ap50_t[valid_cls].numpy()
gt_per_class_ap5095 = ap5095_t[valid_cls].numpy()

# Precision/Recall reali (TP/FP/FN) per le sole classi valutate da torchmetrics
denom_p = (tp_pc + fp_pc).astype(np.float64)
denom_r = (tp_pc + fn_pc).astype(np.float64)
prec_pc = np.divide(tp_pc, denom_p, out=np.zeros_like(denom_p), where=denom_p > 0)
rec_pc  = np.divide(tp_pc, denom_r, out=np.zeros_like(denom_r), where=denom_r > 0)
gt_per_class_p = prec_pc[gt_class_indices]
gt_per_class_r = rec_pc[gt_class_indices]

precision = float(gt_per_class_p.mean()) if len(gt_per_class_p) > 0 else 0.0
recall    = float(gt_per_class_r.mean()) if len(gt_per_class_r) > 0 else 0.0
f1_score  = 2 * precision * recall / (precision + recall + 1e-9)

GT_EVAL_SAVE_DIR = DIR_GT_EVAL   # nessuna confusion matrix automatica con torchmetrics

print(f"\nmAP@0.5:       {map50:.4f}")
print(f"mAP@0.5:0.95:  {map5095:.4f}")
print(f"Precision:     {precision:.4f}  (TP/FP/FN, IoU {MATCH_IOU}, conf {CONF_THRESHOLD})")
print(f"Recall:        {recall:.4f}  (TP/FP/FN, IoU {MATCH_IOU}, conf {CONF_THRESHOLD})")
print(f"F1 Score:      {f1_score:.4f}")
print(f"Classi valutate: {len(gt_class_indices)}")

In [ ]:
# ============================================================
# Cella 8 - Metriche per classe + grafici GT (EfficientDet)
# ============================================================
def _boss_label(model_id):
    bid = MODEL_TO_BOSS.get(int(model_id))
    return BOSS_CLASSES[bid] if bid is not None else MODEL_CLASSES.get(int(model_id), str(model_id))

gt_class_names  = [_boss_label(i) for i in gt_class_indices]
gt_per_class_f1 = 2 * (gt_per_class_p * gt_per_class_r) / (gt_per_class_p + gt_per_class_r + 1e-9)

df_metrics = pd.DataFrame({
    "Classe":      gt_class_names,
    "AP@0.5":      gt_per_class_ap50,
    "AP@0.5:0.95": gt_per_class_ap5095,
    "Precision":   gt_per_class_p,
    "Recall":      gt_per_class_r,
    "F1":          gt_per_class_f1,
})
summary_row = pd.DataFrame([{
    "Classe":      "MEDIA",
    "AP@0.5":      df_metrics["AP@0.5"].mean(),
    "AP@0.5:0.95": df_metrics["AP@0.5:0.95"].mean(),
    "Precision":   df_metrics["Precision"].mean(),
    "Recall":      df_metrics["Recall"].mean(),
    "F1":          df_metrics["F1"].mean(),
}])
df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)

float_cols = ["AP@0.5", "AP@0.5:0.95", "Precision", "Recall", "F1"]
df_display = df_metrics.copy()
df_display[float_cols] = df_display[float_cols].applymap(lambda x: f"{x:.4f}")
print("=== Metriche per Classe (EfficientDet Lite4) ===")
print(df_display.to_string(index=False))
df_metrics.to_csv(DIR_GT_EVAL / "metrics_per_class_gt.csv", index=False)
print(f"\nSalvato: {DIR_GT_EVAL}/metrics_per_class_gt.csv")

df_plot = df_metrics[df_metrics["Classe"] != "MEDIA"].copy()

# Grafico 1: AP@0.5 e AP@0.5:0.95 per classe
x = np.arange(len(df_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, df_plot["AP@0.5"],      width, label="AP@0.5",      color="steelblue")
bars2 = ax.bar(x + width/2, df_plot["AP@0.5:0.95"], width, label="AP@0.5:0.95", color="coral")
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}",
            ha="center", va="bottom", fontsize=8)
ax.set_title("AP per classe - Ground Truth (EfficientDet Lite4)", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(df_plot["Classe"], rotation=45, ha="right")
ax.set_ylabel("AP")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
save_path = DIR_GT_EVAL / "plot_ap_per_class_gt.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Salvato: {save_path}")

# Grafico 2: scatter Precision vs Recall per classe
fig, ax = plt.subplots(figsize=(10, 7))
for _, row in df_plot.iterrows():
    ax.scatter(row["Recall"], row["Precision"], s=100, zorder=5)
    ax.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision vs Recall per classe - Ground Truth (EfficientDet Lite4)")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_path = DIR_GT_EVAL / "plot_pr_scatter_gt.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 9 - Griglia campione frame annotati
# ============================================================
N_SAMPLES = 9

if PREDICT_SAVE_DIR is None or not PREDICT_SAVE_DIR.exists():
    print(f"Cartella inferenza non disponibile: {PREDICT_SAVE_DIR}")
    annotated_imgs = []
else:
    annotated_imgs = (
        list(PREDICT_SAVE_DIR.glob("*.jpg")) +
        list(PREDICT_SAVE_DIR.glob("*.png"))
    )

if len(annotated_imgs) == 0:
    print("Nessun frame annotato disponibile per la griglia.")
else:
    sample = random.sample(annotated_imgs, min(N_SAMPLES, len(annotated_imgs)))
    grid_size = int(np.ceil(np.sqrt(len(sample))))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(5 * grid_size, 4 * grid_size))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    for ax, img_path in zip(axes_flat, sample):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=7)
        ax.axis("off")
    for ax in list(axes_flat)[len(sample):]:
        ax.axis("off")

    plt.suptitle("Campione frame recordings - predizioni B.O.S.S.", fontsize=14)
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_sample_predictions.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 10 - Dashboard riepilogo finale (confronto GT vs modello)
# ============================================================
fig = plt.figure(figsize=(16, 10))
fig.suptitle("B.O.S.S. - Riepilogo Test su recordings (EfficientDet Lite4)", fontsize=16, fontweight="bold")

# Pannello 1: metriche aggregate GT come barre orizzontali
ax1 = fig.add_subplot(2, 3, 1)
metrics_summary = {
    "mAP@0.5":      map50,
    "mAP@0.5:0.95": map5095,
    "Precision":    precision,
    "Recall":       recall,
    "F1 Score":     f1_score,
}
colors_gauge = ["#2196F3", "#1976D2", "#4CAF50", "#FF9800", "#9C27B0"]
bars_h = ax1.barh(list(metrics_summary.keys()), list(metrics_summary.values()), color=colors_gauge)
for bar, val in zip(bars_h, metrics_summary.values()):
    ax1.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=10, fontweight="bold")
ax1.set_xlim(0, 1.1)
ax1.set_title("Metriche Aggregate vs GT", fontsize=11)
ax1.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="soglia 0.5")
ax1.legend(fontsize=8)

# Pannello 2: AP@0.5 per classe
ax2 = fig.add_subplot(2, 3, 2)
ax2.barh(df_plot["Classe"], df_plot["AP@0.5"], color="steelblue")
ax2.set_title("AP@0.5 per Classe", fontsize=11)
ax2.set_xlim(0, 1)
ax2.axvline(map50, color="red", linestyle="--", alpha=0.7, label=f"media={map50:.2f}")
ax2.legend(fontsize=8)

# Pannello 3: F1 per classe
ax3 = fig.add_subplot(2, 3, 3)
ax3.barh(df_plot["Classe"], df_plot["F1"], color="coral")
ax3.set_title("F1 Score per Classe", fontsize=11)
ax3.set_xlim(0, 1)
ax3.axvline(f1_score, color="red", linestyle="--", alpha=0.7, label=f"media={f1_score:.2f}")
ax3.legend(fontsize=8)

# Pannello 4: scatter Precision vs Recall per classe
ax4 = fig.add_subplot(2, 3, 4)
scatter_colors = plt.cm.tab20.colors[:len(df_plot)]
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax4.scatter(row["Recall"], row["Precision"],
                color=scatter_colors[i % len(scatter_colors)], s=100, zorder=5)
    ax4.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                 textcoords="offset points", xytext=(5, 3), fontsize=7)
ax4.set_xlabel("Recall")
ax4.set_ylabel("Precision")
ax4.set_title("Precision vs Recall per Classe", fontsize=11)
ax4.set_xlim(0, 1.05)
ax4.set_ylim(0, 1.05)
ax4.grid(True, alpha=0.3)

# Pannello 5: distribuzione classi BOSS rilevate sui frame recordings
ax5 = fig.add_subplot(2, 3, 5)
df_boss5 = df_preds.dropna(subset=["boss_class"]) if not df_preds.empty else df_preds
if not df_preds.empty and not df_boss5.empty:
    class_counts_test = df_boss5["boss_class"].value_counts()
    ax5.pie(class_counts_test.values,
            labels=class_counts_test.index,
            autopct="%1.1f%%",
            startangle=140,
            textprops={"fontsize": 7})
    ax5.set_title("Distribuzione classi BOSS - Frame recordings", fontsize=11)
else:
    ax5.text(0.5, 0.5, "Nessuna predizione mappata", ha="center", va="center")
    ax5.axis("off")

# Pannello 6: info configurazione
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis("off")
frames_count = len(all_frames)
pred_count   = len(df_preds)
pred_mapped  = int(df_preds["boss_class"].notna().sum()) if not df_preds.empty else 0
gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))
info_text = (
    f"Modello:        {Path(str(MODEL_PATH)).name}\n"
    f"Classi modello: {MODEL_NC}\n"
    f"Classi BOSS:    {NUM_CLASSES}\n"
    f"Input:          {INPUT_H}x{INPUT_W}\n"
    f"Conf threshold: {CONF_THRESHOLD}\n"
    f"IoU threshold:  {IOU_THRESHOLD}\n"
    f"Device:         {DEVICE}\n\n"
    f"Frame recordings:  {frames_count}\n"
    f"Pred. totali:      {pred_count}\n"
    f"Pred. mappate BOSS:{pred_mapped}\n"
    f"GT immagini test:  {gt_img_count}\n"
    f"Output:            TEST_OUTPUT/\n"
)
ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
ax6.set_title("Configurazione", fontsize=11)

plt.tight_layout()
save_path = DIR_COMPARISON / "plot_dashboard_riepilogo.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 11 - Report Markdown (output leggibile da IA)
# ============================================================
def df_to_md(df):
    """Converte un DataFrame in tabella Markdown, senza dipendenze esterne."""
    cols = list(df.columns)
    head = "| " + " | ".join(str(c) for c in cols) + " |"
    sep  = "| " + " | ".join("---" for _ in cols) + " |"
    rows = []
    for _, r in df.iterrows():
        cells = [f"{r[c]:.4f}" if isinstance(r[c], float) else str(r[c]) for c in cols]
        rows.append("| " + " | ".join(cells) + " |")
    return "\n".join([head, sep] + rows)

gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))

if not df_preds.empty:
    class_counts_md = (
        df_preds.dropna(subset=["boss_class"])["boss_class"]
        .value_counts()
        .reindex(BOSS_CLASSES, fill_value=0)
    )
    dist_df = pd.DataFrame({"Classe": class_counts_md.index, "Rilevamenti": class_counts_md.values})
    dist_md = df_to_md(dist_df)
    pred_total       = len(df_preds)
    frames_with_pred = df_preds["frame"].nunique()
else:
    dist_md = "_Nessuna predizione._"
    pred_total = 0
    frames_with_pred = 0

report = f"""# B.O.S.S. - Report Test EfficientDet Lite4

## 1. Configurazione
| Parametro | Valore |
| --- | --- |
| Modello | {Path(str(MODEL_PATH)).name} |
| Classi modello | {MODEL_NC} |
| Classi BOSS | {NUM_CLASSES} |
| Input modello | {INPUT_H}x{INPUT_W} |
| Conf threshold (inferenza) | {CONF_THRESHOLD} |
| IoU threshold | {IOU_THRESHOLD} |
| Device | {DEVICE} |
| Frame recordings | {len(all_frames)} |
| Immagini GT (test) | {gt_img_count} |

## 2. Metriche aggregate vs Ground Truth
_Mediate sulle classi BOSS coperte dal modello._
| Metrica | Valore |
| --- | --- |
| mAP@0.5 | {map50:.4f} |
| mAP@0.5:0.95 | {map5095:.4f} |
| Precision* | {precision:.4f} |
| Recall* | {recall:.4f} |
| F1 Score | {f1_score:.4f} |

_* Precision = AP@0.5 medio per classe; Recall = MAR@100 medio per classe (proxy torchmetrics)._

## 3. Metriche per classe
{df_to_md(df_metrics)}

## 4. Distribuzione predizioni sui recordings
- Predizioni totali: {pred_total}
- Frame con almeno una predizione: {frames_with_pred} / {len(all_frames)}

{dist_md}
"""

report_path = DIR_MARKDOWN / "report.md"
report_path.write_text(report, encoding="utf-8")
print(f"Report Markdown scritto: {report_path}")

metrics_md_path = DIR_MARKDOWN / "metrics_per_class.md"
metrics_md_path.write_text(
    "# Metriche per classe - Ground Truth (EfficientDet Lite4)\n\n" + df_to_md(df_metrics) + "\n",
    encoding="utf-8",
)
print(f"Tabella metriche scritta: {metrics_md_path}")

In [ ]:
# ============================================================
# Cella 12 - Efficienza: GMACs / GFLOPs / parametri su grafo ONNX
# ============================================================
# Contatore uniforme (onnx-tool, modulo condiviso) per tutti i notebook.
# EfficientDet Lite4 e' TFLite -> serve la versione ONNX alla STESSA risoluzione
# di inferenza (INPUT_H x INPUT_W). GMACs = MACs/1e9 ; GFLOPs = 2*GMACs.
%pip install onnx-tool --quiet

import sys
from pathlib import Path

# --- modulo condiviso boss_eval_utils (stesso contatore di tutti i notebook) ---
BOSS_UTILS_DIR = Path("/kaggle/input/PLACEHOLDER-boss-eval-utils")   # <<< INSERIRE PATH KAGGLE
for _c in [BOSS_UTILS_DIR, Path.cwd(), Path("/kaggle/working"), Path("/home/lorenzoverdura8/BOSS_CODE")]:
    if _c and Path(_c).exists() and str(_c) not in sys.path:
        sys.path.insert(0, str(_c))
import boss_eval_utils as beu

DIR_EFFICIENCY = TEST_OUTPUT / "efficiency"
DIR_EFFICIENCY.mkdir(parents=True, exist_ok=True)

# ONNX di EfficientDet Lite4. Convertire una volta dal .tflite, es.:
#   python -m tf2onnx.convert --tflite 2.tflite --output efficientdet_lite4.onnx --opset 13
# poi caricarlo su Kaggle e indicarne il path qui sotto.
EFFICIENTDET_ONNX = Path("/kaggle/input/PLACEHOLDER-efficientdet-onnx/efficientdet_lite4.onnx")   # <<< INSERIRE PATH ONNX KAGGLE

# Shape input lasciata all'ONNX (export a shape fissa). Se l'ONNX ha dimensioni
# dinamiche, passare input_shape=(...) e input_name="..." secondo il layout
# (TFLite NHWC -> (1, INPUT_H, INPUT_W, 3); ONNX NCHW -> (1, 3, INPUT_H, INPUT_W)).
eff = beu.count_onnx_flops_params(EFFICIENTDET_ONNX)

df_eff = pd.DataFrame([{
    "model":    "EfficientDet Lite4",
    "onnx":     EFFICIENTDET_ONNX.name,
    "input_h":  INPUT_H,
    "input_w":  INPUT_W,
    "params":   eff["params"],
    "params_M": round(eff["params"] / 1e6, 4),
    "gmacs":    round(eff["gmacs"], 4),
    "gflops":   round(eff["gflops"], 4),
}])
eff_csv = DIR_EFFICIENCY / "flops_params.csv"
df_eff.to_csv(eff_csv, index=False)
print(df_eff.to_string(index=False))
print(f"\nSalvato: {eff_csv}")